# 호환 사본 안내
원본 보존용 호환 사본. 상위 폴더 COMPATIBILITY.md를 먼저 확인하세요. 회귀 트리의 기본 중요도는 분산/MSE 감소량입니다. 원문의 Gini/Information Gain 표제를 회귀 결과의 지표 이름으로 해석하지 마세요.

# Feature Importance(특성 중요도)

## 데이터 탐색

In [ ]:
import pandas as pd

import os
from pathlib import Path
file_path = os.environ.get('HOUSING_CSV', '/content/sample_data/california_housing_train.csv')
if not Path(file_path).is_file():
    raise FileNotFoundError('HOUSING_CSV에 California Housing CSV 경로를 지정하세요. COMPATIBILITY.md 참고.')
df = pd.read_csv(file_path)

print(f"Data loaded from: {file_path}")

### 데이터 미리보기 (First 5 Rows)

In [ ]:
display(df.head())

### 데이터 정보 (Data Information - Column types and non-null counts)

In [ ]:
display(df.info())

### 기술 통계 (Descriptive Statistics)

In [ ]:
display(df.describe())

## 트리기반 중요도

### Gini Importance

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
import seaborn as sns

# Features (X) and Target (y)
X = df.drop('median_house_value', axis=1)  # Drop the target column to get features
y = df['median_house_value']  # Target column

# Create a Random Forest Regressor model
# n_estimators: The number of trees in the forest.
# random_state: Controls the randomness of the bootstrapping of the samples and features.
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the model
model.fit(X, y)

# Get feature importances
feature_importances = model.feature_importances_

# Create a DataFrame for better visualization
features_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances
})

# Sort features by importance in descending order
features_df = features_df.sort_values(by='Importance', ascending=False)

# Display the feature importances
print("Gini Importance (Feature Importance from RandomForestRegressor):")
display(features_df)

# Plotting the feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=features_df)
plt.title('Feature Importance using Gini Importance (RandomForestRegressor)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### Information Gain


여기서:

(P(x_i))는 확률 변수 (X)가 (x_i) 값을 가질 확률입니다.
(n)은 확률 변수 (X)가 가질 수 있는 고유한 값들의 총 개수입니다.
(\log_b)는 로그 함수를 나타내며, 밑 (b)는 일반적으로 2 (비트), (e) (내츄럴 유닛 또는 나트), 또는 10으로 사용됩니다.
만약 (P(x_i) = 0)인 경우, (P(x_i) \log_b P(x_i)) 항은 0으로 간주합니다 (이는 (\lim_{p \to 0^+} p \log p = 0) 이기 때문입니다).


$(H(X) = -\sum_{i=1}^{n} P(x_i) \log_b P(x_i))$


In [ ]:
from sklearn.tree import DecisionTreeRegressor
import matplotlib.pyplot as plt
import seaborn as sns

# Features (X) and Target (y) are already defined from previous steps
# X = df.drop('median_house_value', axis=1)
# y = df['median_house_value']

# Create a Decision Tree Regressor model
# random_state for reproducibility
dtree_model = DecisionTreeRegressor(random_state=42)

# Train the model
dtree_model.fit(X, y)

# Get feature importances
feature_importances_dt = dtree_model.feature_importances_

# Create a DataFrame for better visualization
features_df_dt = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances_dt
})

# Sort features by importance in descending order
features_df_dt = features_df_dt.sort_values(by='Importance', ascending=False)

# Display the feature importances
print("Information Gain (Feature Importance from DecisionTreeRegressor):")
display(features_df_dt)

# Plotting the feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=features_df_dt)
plt.title('Feature Importance using Information Gain (DecisionTreeRegressor)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 상관계수 기반 중요도

## 히트맵

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the correlation matrix
correlation_matrix = df.corr()

# Set up the matplotlib figure
plt.figure(figsize=(12, 10))

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(correlation_matrix,
            annot=True, # Show the correlation values on the heatmap
            cmap='coolwarm', # Color map
            fmt=".2f", # Format annotation values to two decimal places
            linewidths=.5) # Add lines between cells

plt.title('Correlation Matrix Heatmap of California Housing Dataset')
plt.show()

## Filter methods

### Variance thresholds

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# Features (X) is already defined from previous steps
# X = df.drop('median_house_value', axis=1)

# Create a VarianceThreshold object
# You can set a threshold. Features with variance lower than this threshold will be removed.
# For example, if threshold=0, it removes all zero-variance features (i.e., features with the same value in all samples).
# Here, we'll use a default threshold of 0 to remove constant features.
selector = VarianceThreshold(threshold=4)

# Fit the selector to the data and transform it
X_selected = selector.fit_transform(X)

# Get the names of the selected features
selected_features_mask = selector.get_support()
selected_feature_names = X.columns[selected_features_mask]

print(f"Original number of features: {X.shape[1]}")
print(f"Number of features after Variance Threshold: {X_selected.shape[1]}")
print("Selected features:")
for feature in selected_feature_names:
    print(f"- {feature}")

# You can also display the DataFrame with selected features
# selected_df = pd.DataFrame(X_selected, columns=selected_feature_names)
# display(selected_df.head())

### Chi-square

$$\chi^2 = \sum \frac{(O_i - E_i)^2}{E_i}$$

여기서:

*   ($\chi^2$)는 카이제곱 통계량입니다.
*   ($O_i$)는 각 범주의 관측 빈도(Observed Frequency)입니다.
*   ($E_i$)는 각 범주의 기대 빈도(Expected Frequency)입니다.

기대 빈도 ($E_i$)는 다음과 같이 계산됩니다:

$$E_i = \frac{(\text{row total}) \times (\text{column total})}{(\text{grand total})}$$

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import KBinsDiscretizer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make copies to avoid modifying the original X and y dataframes directly for this operation
X_chi = X.copy()
y_chi = y.copy()

# --- Preprocessing for Chi-square --- #
# 1. Ensure features are non-negative, as chi2 requires this.
# 'longitude' has negative values, so we shift it to make all values non-negative.
min_longitude = X_chi['longitude'].min()
if min_longitude < 0:
    X_chi['longitude'] = X_chi['longitude'] - min_longitude

# 2. Discretize numerical features (X_chi) into bins (categorical representation).
# Using KBinsDiscretizer with 'quantile' strategy for equal frequency bins
# and 'ordinal' encoder to get integer labels suitable for chi2.
# We set n_bins=5 to create 5 categories for each feature.
discretizer_X = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile', subsample=None, random_state=42)
X_chi_discretized = discretizer_X.fit_transform(X_chi)
X_chi_discretized_df = pd.DataFrame(X_chi_discretized, columns=X_chi.columns)

# 3. Discretize the continuous target (y_chi) into categorical bins.
# Using pd.qcut to get roughly equal-sized bins for the target variable.
# 'labels=False' returns integer indicators for the bins.
y_chi_discretized = pd.qcut(y_chi, q=5, labels=False, duplicates='drop')

# --- Apply SelectKBest with chi2 --- #
# Select top k features. Here, we select 'all' to get scores for all features.
selector_chi2 = SelectKBest(chi2, k='all')
selector_chi2.fit(X_chi_discretized_df, y_chi_discretized)

# Get the chi-square scores for each feature
feature_scores_chi2 = selector_chi2.scores_

# Create a DataFrame for better visualization of scores
features_df_chi2 = pd.DataFrame({
    'Feature': X.columns, # Use original column names for clarity
    'Chi2_Score': feature_scores_chi2
})

# Sort features by their Chi2_Score in descending order
features_df_chi2 = features_df_chi2.sort_values(by='Chi2_Score', ascending=False)

print("\nChi-square Scores for Feature Selection (after discretization and non-negative adjustment):")
display(features_df_chi2)

# --- Plotting the feature scores --- #
plt.figure(figsize=(12, 8))
sns.barplot(x='Chi2_Score', y='Feature', data=features_df_chi2, palette='viridis')
plt.title('Feature Importance using Chi-square (after Discretization)')
plt.xlabel('Chi2 Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### 상호 정보 (Mutual Information)

이산 확률 변수 (X)와 (Y) 사이의 상호 정보 (I(X; Y))는 다음과 같이 정의됩니다:

$$I(X; Y) = \sum_{x \in X} \sum_{y \in Y} P(x, y) \log_2 \left(\frac{P(x, y)}{P(x)P(y)}\right)$$

여기서:

*   ($P(x, y)$)는 (X=x)이고 (Y=y)일 공동 확률 분포입니다.
*   ($P(x)$)는 (X=x)일 주변 확률 분포입니다.
*   ($P(y)$)는 (Y=y)일 주변 확률 분포입니다.

상호 정보는 엔트로피를 이용하여 다음과 같이 표현될 수도 있습니다:

$$I(X; Y) = H(X) - H(X|Y)$$
$$I(X; Y) = H(Y) - H(Y|X)$$
$$I(X; Y) = H(X) + H(Y) - H(X, Y)$$

여기서:

*   ($H(X)$)는 변수 (X)의 엔트로피입니다.
*   ($H(Y)$)는 변수 (Y)의 엔트로피입니다.
*   ($H(X|Y)$)는 (Y)가 주어졌을 때 (X)의 조건부 엔트로피입니다.
*   ($H(Y|X)$)는 (X)가 주어졌을 때 (Y)의 조건부 엔트로피입니다.
*   ($H(X, Y)$)는 (X)와 (Y)의 결합 엔트로피입니다.

In [ ]:
from sklearn.feature_selection import mutual_info_regression
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Features (X) and Target (y) are already defined from previous steps
# X = df.drop('median_house_value', axis=1)
# y = df['median_house_value']

# Calculate mutual information scores
# random_state is used for reproducibility in approximations
mi_scores = mutual_info_regression(X, y, random_state=42)

# Create a DataFrame for better visualization
features_df_mi = pd.DataFrame({
    'Feature': X.columns,
    'Mutual_Info_Score': mi_scores
})

# Sort features by importance in descending order
features_df_mi = features_df_mi.sort_values(by='Mutual_Info_Score', ascending=False)

print("\nMutual Information Scores:")
display(features_df_mi)

# Plotting the feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x='Mutual_Info_Score', y='Feature', data=features_df_mi, palette='mako')
plt.title('Feature Importance using Mutual Information')
plt.xlabel('Mutual Information Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Embbeded Methods
L1, L2, Elastic Net

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Features (X) and Target (y) are already defined from previous steps
X = df.drop('median_house_value', axis=1)  # Drop the target column to get features
y = df['median_house_value']  # Target column

# Standardize features before applying Lasso Regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# --- Demonstrate coefficient shrinkage with varying alpha ---

alphas = np.logspace(-3, 5, 10) # Generate 10 alpha values from 0.001 to 10^5
coefficients = []

print("\n--- Lasso Regression with varying alpha values ---")
for alpha in alphas:
    lasso_model_temp = Lasso(alpha=alpha, random_state=42, max_iter=100000) # Increased max_iter for convergence
    lasso_model_temp.fit(X_scaled_df, y)
    coeffs = lasso_model_temp.coef_
    coefficients.append(coeffs)

    zeroed_features = X.columns[coeffs == 0]
    if len(zeroed_features) > 0:
        print(f"Alpha = {alpha:.4f}: Coefficients that became zero: {list(zeroed_features)}")
    else:
        print(f"Alpha = {alpha:.4f}: No coefficients became zero yet.")

coefficients = np.array(coefficients)

# Plotting the coefficient path
plt.figure(figsize=(14, 8))
for i, feature in enumerate(X.columns):
    plt.plot(alphas, coefficients[:, i], label=feature)

plt.xscale('log')
plt.xlabel('Alpha (Log Scale)')
plt.ylabel('Coefficient Value')
plt.title('Lasso Coefficient Path')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

# --- Original Lasso Regression with chosen alpha (for final feature importance) ---

# Create a Lasso Regressor model with a specific alpha
# The choice of alpha is crucial. Here, we use 0.1 as an example.
lasso_model = Lasso(alpha=0.1, random_state=42)

# Train the model
lasso_model.fit(X_scaled_df, y)

# Get feature coefficients (importances)
feature_coefficients = lasso_model.coef_

# Create a DataFrame for better visualization
features_df_lasso = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': feature_coefficients
})

# Sort features by the absolute value of their coefficient in descending order
features_df_lasso['Abs_Coefficient'] = abs(features_df_lasso['Coefficient'])
features_df_lasso = features_df_lasso.sort_values(by='Abs_Coefficient', ascending=False)

print("\nLasso Regression Coefficients (Feature Importance for alpha=0.1):")
display(features_df_lasso[['Feature', 'Coefficient']])

# Plotting the feature importances for alpha=0.1
plt.figure(figsize=(12, 8))
sns.barplot(x='Coefficient', y='Feature', hue='Feature', data=features_df_lasso, palette='viridis', legend=False)
plt.title('Feature Importance using Lasso Regression (alpha=0.1)')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Reuse variables from the previously executed Lasso cell (IstGDyX8Qc60)
# X_scaled_df, y, alphas, and coefficients should be available in the kernel state

print("Lasso Regression Coefficient Path (re-plotting from previous calculation):")

# Plotting the coefficient path
plt.figure(figsize=(14, 8))
for i, feature in enumerate(X.columns):
    plt.plot(alphas, coefficients[:, i], label=feature)

plt.xscale('log')
plt.xlabel('Alpha (Log Scale)')
plt.ylabel('Coefficient Value')
plt.title('Lasso Coefficient Path (Embedded Method Demonstration)')
plt.axhline(0, color='black', linestyle='--', linewidth=0.8)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

print("\nLasso Path Explanation:")
print("Lasso (Least Absolute Shrinkage and Selection Operator)는 L1 정규화를 사용하여 모델 훈련과 동시에 피처 선택을 수행하는 임베디드 방식의 피처 중요도 방법입니다. 그래프는 'alpha' 값이 증가함에 따라 각 피처의 계수(coefficient)가 어떻게 변화하는지 보여줍니다.\n")
print("주요 특징:")
print("1. **계수 축소 (Shrinkage)**: alpha 값이 커질수록 계수들이 0에 가까워지거나 0이 됩니다. 이는 모델의 복잡성을 줄이고 과적합을 방지합니다.\n")
print("2. **피처 선택 (Feature Selection)**: 특정 alpha 값 이상에서 일부 피처의 계수가 정확히 0이 되는 것을 볼 수 있습니다. 이는 해당 피처가 모델에서 제외된다는 것을 의미하며, Lasso가 자동으로 중요한 피처를 선택하는 역할을 한다는 것을 보여줍니다. 그래프에서 계수가 0에 도달하는 지점이 해당 피처가 모델에서 제외되는 alpha 값입니다.\n")
print("3. **Lasso Path**: 이 곡선들을 'Lasso Path'라고 부르며, 정규화 강도(alpha)에 따른 피처 중요도 변화를 직관적으로 파악하는 데 유용합니다.")

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Features (X) and Target (y) are already defined from previous steps
# X = df.drop('median_house_value', axis=1)
# y = df['median_house_value']

# Standardize features before applying Ridge Regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Create a Ridge Regressor model
# alpha: Constant that multiplies the L2 term. Default is 1.0. A higher alpha means more regularization.
ridge_model = Ridge(alpha=1.0, random_state=42) # You can adjust alpha

# Train the model
ridge_model.fit(X_scaled_df, y)

# Get feature coefficients (importances)
feature_coefficients_ridge = ridge_model.coef_

# Create a DataFrame for better visualization
features_df_ridge = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': feature_coefficients_ridge
})

# Sort features by the absolute value of their coefficient in descending order
features_df_ridge['Abs_Coefficient'] = abs(features_df_ridge['Coefficient'])
features_df_ridge = features_df_ridge.sort_values(by='Abs_Coefficient', ascending=False)

print("\nRidge Regression Coefficients (Feature Importance):")
display(features_df_ridge[['Feature', 'Coefficient']])

# Plotting the feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x='Coefficient', y='Feature', hue='Feature', data=features_df_ridge, palette='rocket', legend=False)
plt.title('Feature Importance using Ridge Regression')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
lasso_model.coef_

# Elastic Net
L1 + L2

In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Features (X) and Target (y) are already defined from previous steps
# X = df.drop('median_house_value', axis=1)
# y = df['median_house_value']

# Standardize features before applying ElasticNet Regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Create an ElasticNet Regressor model
# alpha: Constant that multiplies the L1 + L2 terms. Default is 1.0.
# l1_ratio: The ElasticNet mixing parameter, with 0 <= l1_ratio <= 1.
#           For l1_ratio = 0 the penalty is an L2 penalty. For l1_ratio = 1 it is an L1 penalty.
#           For 0 < l1_ratio < 1, the penalty is a combination of L1 and L2.
elastic_net_model = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42, max_iter=10000) # Adjust alpha and l1_ratio as needed

# Train the model
elastic_net_model.fit(X_scaled_df, y)

# Get feature coefficients (importances)
feature_coefficients_en = elastic_net_model.coef_

# Create a DataFrame for better visualization
features_df_en = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': feature_coefficients_en
})

# Sort features by the absolute value of their coefficient in descending order
features_df_en['Abs_Coefficient'] = abs(features_df_en['Coefficient'])
features_df_en = features_df_en.sort_values(by='Abs_Coefficient', ascending=False)

print("\nElastic Net Regression Coefficients (Feature Importance):")
display(features_df_en[['Feature', 'Coefficient']])

# Plotting the feature importances
plt.figure(figsize=(12, 8))
sns.barplot(x='Coefficient', y='Feature', hue='Feature', data=features_df_en, palette='cubehelix', legend=False)
plt.title('Feature Importance using Elastic Net Regression')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

# PCA & t-SNE

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Features (X) is already defined from previous steps
# X = df.drop('median_house_value', axis=1)

# 1. Standardize the data
# PCA is sensitive to feature scaling, so it's important to scale the data first.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Apply PCA
# Change n_components to 3 as requested by the user
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Create a DataFrame for the PCA results
pca_df = pd.DataFrame(data = X_pca, columns = ['principal component 1', 'principal component 2', 'principal component 3'])

print("Original features shape:", X.shape)
print("Reduced features shape (PCA):", X_pca.shape)

print("\nExplained variance ratio for each principal component:")
print(pca.explained_variance_ratio_)
print(f"Total explained variance by 3 components: {pca.explained_variance_ratio_.sum():.2f}")

# Since we now have 3 components, a 3D plot would be more appropriate for direct visualization.
# For a 2D representation, we can still plot two components or use pair plots.
# For simplicity, let's plot the first two components for visualization as before.
plt.figure(figsize=(10, 8))
sns.scatterplot(x='principal component 1', y='principal component 2', data=pca_df, alpha=0.6)
plt.title('2-Component PCA of California Housing Features (from 3 PCs)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.tight_layout()
plt.show()

# Display the first few rows of the PCA transformed data
print("\nFirst 5 rows of PCA transformed data (3 components):")
display(pca_df.head())

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Features (X) is already defined from previous steps
# X = df.drop('median_house_value', axis=1)

# Standardize the data
# t-SNE is sensitive to feature scaling, so it's important to scale the data first.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply t-SNE
# n_components: Dimension of the embedded space (usually 2 or 3 for visualization)
# random_state: For reproducibility
# perplexity: Balances attention between local and global aspects of your data.
#             Typically between 5 and 50. Smaller values focus on local structure.
# n_iter: Maximum number of iterations for the optimization. Increase if results are not stable.

# Note: t-SNE can be computationally intensive, especially for large datasets.
# For very large datasets, consider sampling or using PCA first to reduce dimensionality.

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=250)
X_tsne = tsne.fit_transform(X_scaled)

# Create a DataFrame for the t-SNE results
tsne_df = pd.DataFrame(data = X_tsne, columns = ['t-SNE component 1', 't-SNE component 2'])

print("Original features shape:", X.shape)
print("Reduced features shape (t-SNE):", X_tsne.shape)

# Plotting the t-SNE results
plt.figure(figsize=(10, 8))
sns.scatterplot(x='t-SNE component 1', y='t-SNE component 2', data=tsne_df, alpha=0.6,
                # Optionally, color points by target variable if available
                # hue=df['median_house_value'], palette='viridis'
               )
plt.title('t-SNE of California Housing Features')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.grid(True)
plt.tight_layout()
plt.show()

# Display the first few rows of the t-SNE transformed data
print("\nFirst 5 rows of t-SNE transformed data:")
display(tsne_df.head())